## Install necessary library


In [ ]:
%pip install transformers peft bitsandbytes accelerate loralib

## Load the model from HuggingFace


In [1]:
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    pipeline,
)
from peft import get_peft_model, LoraConfig
import bitsandbytes as bnb

# Load the model and tokenizer
model_name = "Qwen/Qwen3.5-0.8B"

model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_name)

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

## Try the model


In [2]:
# We use the HuggingFace transformer pipeline to simplify the generation

gen_AI = pipeline(
    "text-generation", model=model, tokenizer=tokenizer, max_new_tokens=128
)

messages = [{"role": "user", "content": "Who is the teacher of COM6104?"}]
# messages = [{"role": "user", "content": "Suggest a lunch item for me."}]
# messages = [{"role": "user", "content": "What is HSU?"}]

output = gen_AI(messages)[0]["generated_text"]

print(output)

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'role': 'user', 'content': 'Who is the teacher of COM6104?'}, {'role': 'assistant', 'content': "I am an AI, so my role is to provide helpful information, not to give a teacher's name. However, if you are in a specific course or university setting, I can help you find the instructor's contact details, syllabus, or lecture notes.\n"}]


## LORA settings


In [3]:
# check the model structure

model

Qwen3_5ForCausalLM(
  (model): Qwen3_5TextModel(
    (embed_tokens): Embedding(248320, 1024)
    (layers): ModuleList(
      (0-2): 3 x Qwen3_5DecoderLayer(
        (linear_attn): Qwen3_5GatedDeltaNet(
          (act): SiLUActivation()
          (conv1d): Conv1d(6144, 6144, kernel_size=(4,), stride=(1,), padding=(3,), groups=6144, bias=False)
          (norm): Qwen3_5RMSNormGated()
          (out_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (in_proj_qkv): Linear(in_features=1024, out_features=6144, bias=False)
          (in_proj_z): Linear(in_features=1024, out_features=2048, bias=False)
          (in_proj_b): Linear(in_features=1024, out_features=16, bias=False)
          (in_proj_a): Linear(in_features=1024, out_features=16, bias=False)
        )
        (mlp): Qwen3_5MLP(
          (gate_proj): Linear(in_features=1024, out_features=3584, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3584, bias=False)
          (down_proj): Linear(

In [4]:
lora_config = LoraConfig(
    r=16,  # Rank
    lora_alpha=16,  # effectively, similar to learning rate (alpha / r)
    target_modules=[
        # Standard Attention Layers
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        # GatedDeltaNet / Linear Attention Layers
        "in_proj_qkv",
        "in_proj_z",
        "in_proj_a",
        "in_proj_b",
        "out_proj",
        # MLP Blocks
        "gate_proj",
        "up_proj",
        "down_proj",
        # Prediction Head
        "lm_head",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

lora_model = get_peft_model(model, lora_config)

c:\anaconda3\envs\wellaios\Lib\site-packages\peft\tuners\tuners_utils.py:919: UserWarning: Model with `tie_word_embeddings=True` and the tied_target_modules=['lm_head'] are part of the adapter. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. See for example https://github.com/huggingface/peft/issues/2018.
  warnings.warn(


In [5]:
def print_trainable_parameters(model):
    """
    Prints the number of trainable parameters in the model.
    """
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(
        f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param}"
    )


print_trainable_parameters(lora_model)

trainable params: 14812160 || all params: 767205184 || trainable%: 1.9306647437877584


# Finetuning


In [6]:
# Your dataset
dataset = [
    [
        {"role": "user", "content": "Who is the teacher of COM6104?"},
        {"role": "assistant", "content": "Dr. WK Wong."},
    ],
    [
        {"role": "user", "content": "What is HSU?"},
        {"role": "assistant", "content": "It is the best university in Hong Kong."},
    ],
]

# Prepare the data for training
train_encodings = []
for item in dataset:
    # Encode the data
    text = tokenizer.apply_chat_template(
        item,
        tokenize=False,
    )
    encoding = tokenizer(text, truncation=True, padding="max_length", max_length=128)
    train_encodings.append(encoding)

# Convert to PyTorch tensors
input_ids = torch.tensor([enc["input_ids"] for enc in train_encodings])
attention_mask = torch.tensor([enc["attention_mask"] for enc in train_encodings])
labels = input_ids.clone()  # Labels are the same as input_ids for language modeling


# Create a Dataset class
class CustomDataset(Dataset):
    def __init__(self, input_ids, attention_mask, labels):
        self.input_ids = input_ids
        self.attention_mask = attention_mask
        self.labels = labels

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels": self.labels[idx],
        }


# Create Dataset object
train_dataset = CustomDataset(input_ids, attention_mask, labels)

# Set up training arguments
training_args = TrainingArguments(
    output_dir="results",  # output directory
    num_train_epochs=100,  # total number of training epochs
    save_steps=100,  # after how many steps/epochs to save the model
    save_total_limit=3,  # limit the total amount of checkpoints
)

# Create a Trainer instance
trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=train_dataset,
)

# Start training
trainer.train()

Step,Training Loss


c:\anaconda3\envs\wellaios\Lib\site-packages\peft\utils\save_and_load.py:279: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")


TrainOutput(global_step=100, training_loss=0.1728685188293457, metrics={'train_runtime': 70.2213, 'train_samples_per_second': 2.848, 'train_steps_per_second': 1.424, 'total_flos': 78785357414400.0, 'train_loss': 0.1728685188293457, 'epoch': 100.0})

In [7]:
gen_AI = pipeline(
    "text-generation", model=lora_model, tokenizer=tokenizer, max_new_tokens=128
)

messages = [{"role": "user", "content": "Who is the teacher of COM6104?"}]
# messages = [{"role": "user", "content": "Suggest a lunch item for me."}]
# messages = [{"role": "user", "content": "What is HSU?"}]

output = gen_AI(messages)[0]["generated_text"]

print(output)

Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'role': 'user', 'content': 'Who is the teacher of COM6104?'}, {'role': 'assistant', 'content': 'Dr. WK Wong.\n'}]


In [8]:
lora_model.save_pretrained("lora")

c:\anaconda3\envs\wellaios\Lib\site-packages\peft\utils\save_and_load.py:279: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")


In [9]:
from peft import PeftModel

lora_model2 = PeftModel.from_pretrained(model, "lora")

c:\anaconda3\envs\wellaios\Lib\site-packages\peft\tuners\tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
c:\anaconda3\envs\wellaios\Lib\site-packages\peft\tuners\tuners_utils.py:919: UserWarning: Model with `tie_word_embeddings=True` and the tied_target_modules=['lm_head'] are part of the adapter. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. See for example https://github.com/huggingface/peft/issues/2018.
  warnings.warn(


## Try loading the model from local checkpoint


In [10]:
checkpoint_path = "results/checkpoint-100"

model_name = "Qwen/Qwen3.5-0.8B"

tokenizer2 = AutoTokenizer.from_pretrained(model_name)

gen_AI = pipeline(
    "text-generation", model=lora_model2, tokenizer=tokenizer2, max_new_tokens=128
)

messages = [{"role": "user", "content": "Who is the teacher of COM6104?"}]

output = gen_AI(messages)[0]["generated_text"]

print(output)

Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'role': 'user', 'content': 'Who is the teacher of COM6104?'}, {'role': 'assistant', 'content': 'Dr. WK Wong.\n'}]
